### Homework 2 (part 1): Finetuning BERT

Your task today will be to play with BERT embedding generation, finetune existing models on new data and behold transformer superiority over previous architectures (even though at the expense of heavier computational costs).

In [39]:
%pip install -q --upgrade transformers datasets accelerate deepspeed tiktoken protobuf sentencepiece tokenizers scikit-learn

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import transformers
import datasets
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments

Note: you may need to restart the kernel to use updated packages.


### Load data and model

Our dataset for today is a **Quora Question Pairs (QQP)**.

The dataset consists of over 400,000 question pairs, and each question pair is annotated with a binary value indicating whether the two questions are paraphrase of each other i.e. semantically close. Read [here](https://paperswithcode.com/dataset/quora-question-pairs) if you want to know more.

In [3]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

### Tokenize the data

The [dataset](https://huggingface.co/docs/datasets/en/index) library allows you to use mapping as in the functional-style programming.

What Happens to the Texts in `qqp_preprocessed`?

- The original `text1` and `text2` are tokenized into numerical ids using a relevant tokenizer.
- Both texts are concatenated via the `SEP` token and are prepended using the `CLS` token in order to meet the required formet. The resulting sequence is either truncated (if combined length > 128 tokens) or padded (if combined length < 128 tokens).
- The `qqp_preprocessed` dataset contains:
    - _Input IDs_: sequence of token ids.
    - _Attention Masks_: binary masks indicating which tokens are padding.
    - _Token Type IDs_: distinguish between tokens from text1 and text2.

__!Note!__ Attention masks here allow skipping computation on `PAD` tokens.

In [4]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [5]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (2 points)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

Just glimpsing at our data

In [6]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [7]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())
# print('\nPrediction (probs):', torch.argmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

Note that the model uses 2 heads for binary classification (one for each class), not one. This is, in fact, a matter of preference.

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


Note that even though the model computation runs on the GPU, the process of loading data from disk (or memory) into the format required by the model (e.g., tensors) is handled by the CPU.

Insufficient CPU computation resources may result in bottlenecking the whole process.

In [8]:
from tqdm import tqdm
import multiprocessing

cores = multiprocessing.cpu_count() # Count the number of cores in a computer
cores

16

In [9]:
# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create a DataLoader for the validation set
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16,  # Larger batch size for faster processing
    shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=cores  # Use multiple workers to load data faster
)

In [ ]:
# Measure validation accuracy
model.eval()  # Set model to evaluation mode

# <YOUR CODE HERE>
# Initialize counters for accuracy calculation
correct_predictions = 0
total_predictions = 0

# Enable mixed precision for faster computation if supported
scaler = torch.amp.GradScaler('cuda') if device == torch.device("cuda") else None

with torch.no_grad():  # Disable gradient calculation
    for batch in tqdm(val_loader, desc="Evaluating"):
        # Move batch to GPU
        batch = {k: v.to(device) for k, v in batch.items()}  # <YOUR CODE HERE>

        # Use mixed precision if available
        if scaler:
            with torch.amp.autocast('cuda'):
                outputs = model(
                    input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    token_type_ids=batch['token_type_ids']
                )  # <YOUR CODE HERE>
        else:
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch['token_type_ids']
            )  # <YOUR CODE HERE>

        # Get predictions and update accuracy
        # <YOUR CODE HERE>
        predictions = torch.argmax(outputs.logits, dim=1) 
        correct_predictions += (predictions == batch['labels']).sum().item()
        total_predictions += batch['labels'].size(0)

# Compute accuracy
accuracy = correct_predictions / total_predictions # <YOUR CODE HERE> # Validation accuracy, between 0 and 1
print(f"Validation Accuracy: {accuracy:.4f}")


Evaluating: 100%|██████████| 2527/2527 [00:35<00:00, 70.40it/s]

Validation Accuracy: 0.9084


In [11]:
assert 0.9 < accuracy < 0.91

### Train the model (3 points)

For this task, you have to fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.

In [ ]:
# Load your model e.g. DeBERTa-v3 tokenizer and model
model_name = "microsoft/deberta-v3-base" # <THE MODEL OF YOUR CHOICE HERE>
tokenizer = AutoTokenizer.from_pretrained(model_name)  # <YOUR CODE HERE>
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2  # Binary classification for QQP
)  # <YOUR CODE HERE>  # Binary classification. num_labels=1 if you prefer.

# Note that if the tokenizer of your model
# is different from the one we used aboVe,
# you need ot preprocess your data again.

# Preprocess the data
# <YOUR CODE HERE>
def preprocess_function_new(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['labels'] = examples['label']  # для Trainer'a 'labels' вместо 'label'
    return result
    

# <If so, your code goes here>
qqp_preprocessed = qqp.map(preprocess_function_new, batched=True)  # <YOUR CODE HERE>

/home/yc-user/jupyter-notebooks/NLP-course/5.BERT_GPT/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 40430/40430 [00:06<00:00, 6718.53 examples/s]


In [13]:
print(repr(qqp_preprocessed['validation'][0]['input_ids'])[:100], "...")

[1, 1167, 281, 2523, 271, 22642, 324, 874, 302, 2, 1167, 281, 97251, 268, 324, 874, 302, 2, 0, 0, 0, ...


In [ ]:
# Prepare the training and validation sets
train_set = qqp_preprocessed['train']
val_set = qqp_preprocessed['validation']  


# Define a metric for evaluation. You can write your own if you prefer
from sklearn.metrics import accuracy_score

# If you are using transformers.Trainer, you may want to use a utility function below
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for the model during training or evaluation.
    Args:
        eval_pred (tuple): A tuple containing:
            - logits (ndarray or torch.Tensor): The raw logits output by the model for each sample
              in the evaluation batch. Shape: (batch_size, num_classes).
            - labels (ndarray or torch.Tensor): The ground truth labels for each sample in the batch.
              Shape: (batch_size,).
    Returns:
        dict: A dictionary containing the computed metric(s):
            - "accuracy" (float): The proportion of correct predictions over the total number of samples.
    """
    # <YOUR CODE HERE>
    logits, labels = eval_pred
    if isinstance(logits, (tuple, list)):
        logits = logits[0]  # Если HF передает logits как tuple
    preds = logits.argmax(axis=-1)
    accuracy = accuracy_score(labels, preds)
    return {"accuracy": accuracy}

# Feel free not to use transformers.Trainer and write the code manually if you want
# A good starting learning rate is 2e-5.
# A step of an order of magnitude is a good way to adjust it if necessary e.g. 2e-4, 2e-3 etc.
# 3 train epochs is likely enough for gently finetuning the model without the model 'forgetting previous data'
# Be sure to use weight_decay i.e. regularisation. A good starting point is 1e-2. Feel free to experiment.
# Consider setting accuracy as the metric for the best model.

# Define your training arguments without the 'device' argument since it is handled automatically.
training_args = TrainingArguments(
    # <YOUR CODE HERE>
    output_dir="deberta-qqp",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=32,  # 8.7GB max VRAM usage on L4
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=1e-2,
    warmup_ratio=0.06,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,  # мы растим accuracy
    fp16=torch.cuda.is_available(),
    logging_steps=2000,
    save_steps=2000,
    save_total_limit=3,
    report_to=[],
    # делаю на ВМ в облаке, и иногда теряю соединение
    resume_from_checkpoint=True  # на случай, если что-то пошло не так
)

# Initialize the Trainer
trainer = Trainer(
    # <YOUR CODE HERE>
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=val_set,
    tokenizer=tokenizer,
    data_collator=transformers.default_data_collator,
    compute_metrics=compute_metrics
)

# Fine-tune the model
trainer.train()  # <YOUR CODE HERE>

# Evaluate the model
# <YOUR CODE HERE>
metrics = trainer.evaluate()
accuracy = metrics.get("eval_accuracy")
print(f"Validation Accuracy: {accuracy:.4f}")


/tmp/ipykernel_50785/1072665296.py:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy
2000,0.116300,0.282084,0.890675
4000,0.208200,0.240987,0.900371
6000,0.242700,0.240623,0.905417
8000,0.233100,0.231762,0.904353
10000,0.226100,0.216546,0.912392
12000,0.197700,0.256558,0.914321
14000,0.149300,0.240546,0.915113
16000,0.147500,0.218236,0.918897
18000,0.145800,0.228616,0.918105
20000,0.143700,0.217279,0.921593


Validation Accuracy: 0.9250


In [28]:
assert 0.9 < accuracy

To be completely honest, we made a small crime here. Validation part of the dataset is intended for tuning the hyperparameters, but for the sake of simplicity we ommited that logic here. You are free to pick the best hyperparameters and test the results on the `test` subsample if you feel so.

### BONUS: Get a taste of how BERT embeddings work

It is time to shed light on how a BERT-based embedder can be leveraged in searching relevant information.

The problem with vanilla BERT and the likes is that it isn't directly trained using contrastive or triplet loss in order to genuinely force similar embeddings closer to each other. Hence, to obtain the best possible results in building a search engine it is preferrable to pick a dedicated [sentence similarity](https://huggingface.co/models?pipeline_tag=sentence-similarity) model. Feel free to pick the one that will likely meet your requirements the most.

Similar to what we showcased in the first homework, your task is to construct a search engine:
1) _Prepare an embeddings database_: Since Quora Question Pairs dataset contains, well, pairs of questions, we will only pick data in the `text1` field of the `validation` subsample. You should obtain embeddings using a model of your choice and store them for later use in a `numpy.ndarray`. Optionally, you can leverage a dedicated [Faiss](https://github.com/facebookresearch/faiss) index.
2) _Implement a way to search for similar questions to a given query_: It is expected that you will write a function or a class to streamline interactions with your database. __A completion of this part of the homework will be judged upon the ability to print coherently the TOP 5 most similart quora questions given a new arbitrary query.__

Hopefully, you can appreciate how the search has become more semantically profound as compared to our previous attempt.

In [33]:
# Make sure to run Cells 1 and 2 before.

# Initialize the model and its tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"  # <MODEL OF YOUR CHOICE HERE>
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Модель `{model_name}` загружена на `{device.type}`")
# <YOUR CODE HERE>

Модель `sentence-transformers/all-MiniLM-L6-v2` загружена на `cuda`


In [36]:
# Шаг 2: Создание базы данных embeddings
import numpy as np
from tqdm import tqdm

print("Создание базы данных embeddings...")
print(f"Размер validation датасета: {len(qqp['validation'])}")

# Извлекаем все text1 из validation датасета
validation_questions = []
for item in qqp['validation']:
    validation_questions.append(item['text1'])
# Создаем embeddings для всех вопросов

print("Генерация embeddings...")

# Используем batch encoding для эффективности
batch_size = 32
embeddings_list = []

# from https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


# Обрабатываем вопросы батчами
for i in tqdm(range(0, len(validation_questions), batch_size)):
    batch_questions = validation_questions[i:i + batch_size]
    
    # Tokenize sentences
    encoded_input = tokenizer(batch_questions, padding=True, truncation=True, return_tensors='pt')

    # Получаем embeddings для батча
    with torch.no_grad():
        model_output = model(**(encoded_input.to(device)))

        # Perform pooling
        batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

    embeddings_list.append(batch_embeddings.to('cpu'))

# Объединяем все embeddings в один массив
embeddings_database = np.vstack(embeddings_list)

# Нормализуем embeddings для косинусного сходства (как в первом домашнем задании)
embeddings_database_normalized = embeddings_database / np.linalg.norm(embeddings_database, axis=1, keepdims=True)

print(f"Размер базы embeddings: {embeddings_database_normalized.shape}")
print(f"Тип данных: {embeddings_database_normalized.dtype}")

Создание базы данных embeddings...
Размер validation датасета: 40430
Генерация embeddings...


100%|██████████| 1264/1264 [00:08<00:00, 153.36it/s]


Размер базы embeddings: (40430, 384)
Тип данных: float32


In [37]:
# Шаг 3: Реализация функции поиска похожих вопросов
# Я тут добавил tokenizer и исходную базу вопросов в которой ищем
def find_similar_questions(query, database, model, tokenizer, questions_list, top_k = 5, quiet = False):
    """
    Finds and prints the top_k most similar questions for a query.

    This function encodes a query, compares it against a pre-computed
    embedding database using cosine similarity, and prints the most
    semantically similar questions.

    Args:
        query (str): The user's search query.
        database (np.ndarray): A 2D NumPy array containing the pre-computed
                               embeddings for the database of questions.
        model (SentenceTransformer): The initialized Sentence-Transformer model
                                     used to encode the query.
        top_k (int): The number of top results to display.

    Returns:
        None. The function prints the results.
    """
    # <YOUR CODE HERE>        

    # Кодируем запрос
    encoded_input = tokenizer(query, padding=True, truncation=True, return_tensors='pt')

    # Вычисляем embedding
    with torch.no_grad():
        model_output = model(**(encoded_input.to(device)))
        query_embedding = mean_pooling(model_output, encoded_input['attention_mask'])
        query_embedding = query_embedding.to('cpu').numpy()
    
    # Нормализуем ...
    query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)

    # Убираем лишнее измерение если есть
    if query_embedding_normalized.ndim > 1:
        query_embedding_normalized = query_embedding_normalized.squeeze()

    # Вычисляем косинусное сходство
    similarities = np.dot(database, query_embedding_normalized)
    
    # Находим индексы топ-k наиболее похожих вопросов
    top_k_indices = np.argsort(similarities)[::-1][:top_k]
    top_k_similarities = similarities[top_k_indices]
    
    # Выводим результаты
    if not quiet:
        print(f"Запрос: {query}")
        print(f"\nТоп-{top_k} наиболее похожих вопросов:")
        print("-" * 80)
        
        for i, (idx, similarity) in enumerate(zip(top_k_indices, top_k_similarities), 1):
            print(f"{i}. {questions_list[idx]}")
            print(f"   Сходство: {similarity:.4f}")
            print()
    
    return top_k_indices, top_k_similarities

In [38]:
# Тест 1
find_similar_questions(
    "How to learn machine learning?", 
    embeddings_database_normalized, 
    model, 
    tokenizer,
    validation_questions, 
    top_k=5
)

Запрос: How to learn machine learning?

Топ-5 наиболее похожих вопросов:
--------------------------------------------------------------------------------
1. How do I learn machine learning?
   Сходство: 0.9852

2. How do I learn machine learning?
   Сходство: 0.9852

3. How do I learn machine learning?
   Сходство: 0.9852

4. How do I learn machine learning?
   Сходство: 0.9852

5. How can I learn machine learning?
   Сходство: 0.9811



(array([ 9372, 13495, 27107, 20266, 39039]),
 array([0.98522735, 0.98522735, 0.98522735, 0.98522735, 0.9811356 ],
       dtype=float32))

In [9]:
# Тест 2
find_similar_questions(
    "What are the benefits of exercise?", 
    embeddings_database_normalized, 
    model, 
    tokenizer,
    validation_questions, 
    top_k=5
)

Запрос: What are the benefits of exercise?

Топ-5 наиболее похожих вопросов:
--------------------------------------------------------------------------------
1. What are the joys of exercise?
   Сходство: 0.7557

2. What are the benefits of squats?
   Сходство: 0.6437

3. What can I do to maintain my motivation to exercise?
   Сходство: 0.6203

4. What are the advantage of yoga therapy?
   Сходство: 0.6009

5. What are the benefits of swimming weekly twice for one hour?
   Сходство: 0.5863



(array([27975,  2363, 19915,  3801, 17944]),
 array([0.7556565 , 0.64368534, 0.6202915 , 0.60094434, 0.5863292 ],
       dtype=float32))

In [ ]:
# Тест 3: Найдем вопрос, который точно есть в датасете
sample_question = validation_questions[100]
find_similar_questions(
    sample_question, 
    embeddings_database_normalized, 
    model, 
    tokenizer,
    validation_questions, 
    top_k=5
)

Запрос: Which kind of fit can we say of pen and their opener (cap)?

Топ-5 наиболее похожих вопросов:
--------------------------------------------------------------------------------
1. Which kind of fit can we say of pen and their opener (cap)?
   Сходство: 1.0000

2. Which are the best fountain pens?
   Сходство: 0.5337

3. What is the best brand of pen drive?
   Сходство: 0.5121

4. What is pen?
   Сходство: 0.4466

5. What can I substitute for a guitar capo?
   Сходство: 0.4285



(array([  100, 29422, 19030, 37749, 14190]),
 array([0.9999999 , 0.53366315, 0.5121336 , 0.44660413, 0.4284502 ],
       dtype=float32))

# Let's go wild :)
(with a little help of... you know)

In [ ]:
# Установка Gradio для интерактивного веб-интерфейса
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gradio as gr
import pandas as pd

# Функция для Gradio интерфейса
def search_questions_gradio(query, top_k=5):
    """
    Функция поиска для Gradio интерфейса
    """
    if not query.strip():
        return "Пожалуйста, введите запрос для поиска."
    
    try:
        # Используем нашу функцию поиска
        indices, similarities = find_similar_questions(
            query, 
            embeddings_database_normalized, 
            model, 
            tokenizer,
            validation_questions, 
            top_k=top_k,
            quiet = True  # чтобы не выводила лишнего
        )
        
        # Формируем результат для отображения
        results = []
        for i, (idx, similarity) in enumerate(zip(indices, similarities), 1):
            results.append({
                "Ранг": i,
                "Вопрос": validation_questions[idx],
                "Сходство": f"{similarity:.4f}"
            })
        
        # Создаем DataFrame для красивого отображения
        df = pd.DataFrame(results)
        return df
        
    except Exception as e:
        return f"Ошибка при поиске: {str(e)}"


# Создание Gradio интерфейса
def create_gradio_interface():
    """
    Создает интерактивный веб-интерфейс для поиска вопросов
    """
    
    # Создаем интерфейс
    with gr.Blocks(
        title="🔍 Поиск похожих вопросов Quora",
        theme=gr.themes.Soft(),
        css="""
        .gradio-container {
            max-width: 1200px !important;
        }
        .main-header {
            text-align: center;
            margin-bottom: 20px;
        }
        """
    ) as interface:
        
        # Заголовок
        gr.HTML("""
        <div class="main-header">
            <h1>🔍 Поиск похожих вопросов Quora</h1>
            <p>Семантический поиск на основе BERT embeddings</p>
            <p>База данных: 40,430 вопросов из Quora Question Pairs</p>
        </div>
        """)
        
        with gr.Row():
            with gr.Column(scale=3):
                # Поле ввода запроса
                query_input = gr.Textbox(
                    label="🔍 Введите ваш запрос",
                    placeholder="Например: How to learn machine learning?",
                    lines=2,
                    max_lines=4
                )
                
                # Слайдер для количества результатов
                top_k_slider = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=5,
                    step=1,
                    label="📊 Количество результатов",
                    info="Выберите количество похожих вопросов для отображения"
                )
                
                # Кнопка поиска
                search_button = gr.Button(
                    "🚀 Найти похожие вопросы",
                    variant="primary",
                    size="lg"
                )
                
                # Примеры запросов
                gr.HTML("""
                <div style="margin-top: 20px;">
                    <h4>💡 Примеры запросов:</h4>
                    <ul>
                        <li>How to learn machine learning?</li>
                        <li>What are the benefits of exercise?</li>
                        <li>How to get a job in tech industry?</li>
                        <li>What is the meaning of life?</li>
                        <li>Best programming languages to learn</li>
                    </ul>
                </div>
                """)
            
            with gr.Column(scale=7):
                # Область результатов
                results_output = gr.Dataframe(
                    label="📋 Результаты поиска",
                    headers=["Ранг", "Вопрос", "Сходство"],
                    datatype=["number", "str", "str"],
                    interactive=False,
                    wrap=True
                )
        
        # Обработчик событий
        search_button.click(
            fn=search_questions_gradio,
            inputs=[query_input, top_k_slider],
            outputs=results_output
        )
        
        # Обработчик для Enter в поле ввода
        query_input.submit(
            fn=search_questions_gradio,
            inputs=[query_input, top_k_slider],
            outputs=results_output
        )
        
        # Информация о системе
        gr.HTML(f"""
        <div style="margin-top: 30px; padding: 20px; border-radius: 10px;">
            <h4>ℹ️ Информация о системе:</h4>
            <ul>
                <li><strong>Модель:</strong> {model_name}</li>
                <li><strong>Размерность embeddings:</strong> {embeddings_database_normalized.shape[1]}</li>
                <li><strong>База данных:</strong> {len(validation_questions)} вопросов из QQP validation</li>
                <li><strong>Метрика сходства:</strong> Косинусное сходство</li>
                <li><strong>Нормализация:</strong> L2 нормализация для эффективности</li>
            </ul>
        </div>
        """)
    
    return interface


In [25]:
# Ллокальный запуск (без публичной ссылки)
print("Альтернативный запуск (локальный)...")

# Создаем интерфейс
interface = create_gradio_interface()

# Запускаем локально
interface.launch(
    share=False,  # Только локальный доступ
    server_name="127.0.0.1",  # Локальный адрес
    server_port=7861,  # Другой порт
    show_error=True,
    quiet=False,
    inbrowser=True
)

print("Локальный интерфейс запущен на http://127.0.0.1:7861")
print("Чтобы остановить и освободить порт выполните ячейку ниже")

Альтернативный запуск (локальный)...
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Локальный интерфейс запущен на http://127.0.0.1:7861
Чтобы остановить и освободить порт выполните ячейку ниже


In [26]:
interface.close()

Closing server running on port: 7861
